<a href="https://colab.research.google.com/github/Anubarak123/Disaster-Information-Collection-System/blob/main/event_insight_chained_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Event Insight — Integrated Pipeline (Workshop 1-6 Combined)
Environment Setup → Intent Clarification → Event Identification → Image Collection → Summarization → Validation

This notebook integrates all six workshop notebooks into one end-to-end pipeline.
Model: gemini-2.5-flash (unified across all Gemini calls)



## Workshop 1 — Environment Setup
Verifies Google Gemini API connectivity (basic + web search grounding) and Wikimedia API image search.

In [ ]:
# -*- coding: utf-8 -*-
"""event_insight_chained_clean.ipynb
Automatically generated by Colab.
Original file is located at
    https://colab.research.google.com/drive/1nRQ1qlpSISQUVPGZv-vxRwJfEKhfUGjb
Event Insight — Integrated Pipeline (Workshop 1-5 Combined)
Environment Setup → Intent Clarification → Intent Clarification → Event Identification → Image Collection → Summarization
Validation schemas & pydantic imports are placed in Workshop1 for early availability
Model: gemini-2.5-flash (single client initialized once here)
"""
import os
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["LC_ALL"] = "en_US.UTF-8"
os.environ["LANG"] = "en_US.UTF-8"
import json
import requests
from google import genai
from google.genai import types
from google.colab import userdata

# Pydantic imported ONCE in Workshop1 for whole pipeline
!pip install pydantic --quiet
import pydantic
from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import List
import typing_extensions as te

# --------------------------
# All Pydantic Schema Definitions (moved from original Workshop6 Step2)
# Agent data‑contract schemas for runtime boundary validation
# --------------------------
class IntentOutput(BaseModel):
    """Schema for Intent Clarification Agent output."""
    event_type: str
    location: str
    time_period: str
    # min_length=1 enforces non‑empty list
    keywords: List[str] = Field(min_length=1)

class ArticleItem(BaseModel):
    """Nested schema: single news article inside EventOutput."""
    title: str
    url: str
    snippet: str

class EventOutput(BaseModel):
    """Schema for Event Identification Agent output."""
    event_name: str
    event_type: str
    location: str
    date: str
    casualties: str
    summary: str
    severity_score: int
    # Nested list: Pydantic recursively validates every ArticleItem
    articles: List[ArticleItem]

class ImageItem(BaseModel):
    """Schema for one collected image record."""
    title: str
    url: str
    thumbnail_url: str
    description: str
    license: str
    author: str
    local_path: str

class SituationBrief(BaseModel):
    """Schema for Summarization Agent output."""
    headline: str
    overview: str
    severity: str
    affected_population: str
    response: str
    key_facts: List[str]
    # Require at least one source entry
    sources_used: List[str] = Field(min_length=1)

    @field_validator("headline")
    @classmethod
    def headline_max_20_words(cls, v: str) -> str:
        """Custom validator: reject headline longer than 20 words."""
        word_count = len(v.split())
        if word_count > 20:
            raise ValueError(f"Headline exceeds max 20 words, got {word_count}")
        return v

# --------------------------
# Validation helper functions moved from original Workshop6 Step5‑7
# Boundary validation utilities for each agent
# --------------------------
def validated_intent_agent_output(raw_json_str: str):
    """Parse and validate the intent agent's JSON output.
    Returns a dict on success, None if validation fails (caller should re‑prompt).
    """
    try:
        data = json.loads(raw_json_str)
        validated = IntentOutput.model_validate(data)
        return validated.model_dump()
    except ValidationError as e:
        print(f"Intent validation error: {e}")
        return None

def validated_event_agent_output(raw_json_str: str) -> dict:
    """Parse and validate the event agent's JSON output.
    Raises ValueError if validation fails — a bad event output is unrecoverable.
    """
    raw = raw_json_str.strip()
    if raw.startswith("```"):
        raw = raw.removeprefix("```").removesuffix("```").strip()
        if raw.startswith("json"):
            raw = raw.removeprefix("json").strip()
    try:
        data = json.loads(raw)
        validated = EventOutput.model_validate(data)
        return validated.model_dump()
    except ValidationError as e:
        raise ValueError(f"Event agent validation failed: {e}")

def validated_image_item(raw_image_dict: dict):
    """Validate a single image dict against the ImageItem schema.
    Returns a validated dict on success, None on failure.
    """
    try:
        validated = ImageItem.model_validate(raw_image_dict)
        return validated.model_dump()
    except ValidationError as e:
        print(f"Warning: skip invalid image, error: {e}")
        return None

def validate_images_list(images: list) -> list:
    """Validate a list of raw image dicts, skipping any that fail."""
    output = []
    for img in images:
        res = validated_image_item(img)
        if res is not None:
            output.append(res)
    return output

def validated_situation_brief(raw_json_str: str) -> dict:
    """Parse and validate the summarization agent's JSON output.
    Raises ValueError if validation fails.
    """
    raw = raw_json_str.strip()
    if raw.startswith("```"):
        raw = raw.removeprefix("```").removesuffix("```").strip()
        if raw.startswith("json"):
            raw = raw.removeprefix("json").strip()
    try:
        data = json.loads(raw)
        validated = SituationBrief.model_validate(data)
        return validated.model_dump()
    except ValidationError as e:
        raise ValueError(f"SituationBrief validation failed: {e}")

# --------------------------
# Gemini client initialized ONCE in Workshop1, reused across whole notebook
# --------------------------
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

# --- Test 1: Basic Gemini response ---
print("1. Testing Gemini API...")
r1 = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Reply in one sentence confirming you are working correctly."
)
print(f"   ✓ {r1.text.strip()}\n")

# --- Test 2: Gemini with Google Search grounding ---
print("2. Testing Gemini web search...")
r2 = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="List the 3 most recent major natural disaster events with their names and dates only.",
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())]
    )
)
print(f"   ✓ Search response:\n{r2.text.strip()}\n")

# --- Test 3: Wikimedia image search (fixed 404 endpoint, FIXED ASCII User‑Agent hyphen bug) ---
print("3. Testing Wikimedia API...")
# FIX: Use normal ASCII hyphen '-', NOT unicode en‑dash '\u2011' in User‑Agent header
r3 = requests.get(
    "https://commons.wikimedia.org/w/api.php",
    params={
        "action": "query",
        "list": "search",
        "srnamespace": "6",  # 6 represents the file/image namespace
        "srsearch": "2023 Turkey earthquake",
        "srlimit": 2,
        "format": "json"
    },
    headers={"User-Agent": "event-insight/1.0"}
)
r3.raise_for_status()
data = r3.json()
files = data["query"]["search"]
print(f"   ✓ {len(files)} images found")
for f in files:
    print(f"      {f.get('title')}")

print("\n✓ All API connections working — ready for Workshop 2.")

"""Issue 1: Gemini Model Quota Error
The original code provided by the instructor used the Gemini 2.0 Flash model. When running the code, the API returned a 429 RESOURCE_EXHAUSTED error because the current API key did not have available quota for the Gemini 2.0 Flash model.
The error message showed that the quota limit for the model was 0, which means the request was rejected due to model‑specific quota restrictions rather than a problem with the API key, SDK, or code structure.
Solution:
The model was updated from Gemini 2.0 Flash to Gemini 2.5 Flash . The API key was already verified to have access to Gemini 2.5 models, so switching the model resolved the quota issue while keeping the same Gemini API workflow.

Issue 2: Wikimedia API Endpoint Error
The original code used the Wikimedia Commons API endpoint:
https://api.wikimedia.org/core/v1/commons/search/files
This endpoint returned errors because the API format and access requirements were different from the current environment. The issue was related to the API endpoint compatibility rather than the network connection or request logic.
Solution:
The Wikimedia request was changed to use the standard MediaWiki API endpoint:
https://commons.wikimedia.org/w/api.php
The updated request searches the Wikimedia Commons file namespace directly by using the MediaWiki search API. This endpoint works without additional authentication and is more stable for this type of image search task.

Issue3: UnicodeEncodeError Wikimedia User‑Agent header
Root cause: Non‑ASCII unicode en‑dash character U+2011 inside User‑Agent header. Requests http headers strictly require ASCII characters.
Solution: Replace unicode en‑dash with standard ASCII hyphen '-' in User‑Agent string.

After these adjustments, the Gemini API and Wikimedia API connections were successfully restored, and the environment is ready for Workshop 2.
"""


1. Testing Gemini API...
   ✓ I am functioning correctly.

2. Testing Gemini web search...
   ✓ Search response:
The three most recent major natural disaster events are:

1.  **Mount Anak Krakatau Eruption, Indonesia**: September 5-6, 2026. This eruption caused ash clouds, leading to the temporary closure of multiple airports, including Soekarno Hatta and Halim Airports, disrupting air travel.
2.  **Typhoon Halong, Japan**: September 4, 2026. Described as a "Violent Typhoon," it was on a direct path toward Japan's Izu Islands and posed a significant threat to the greater Tokyo region.
3.  **Southern France Flooding**: September 4, 2026. A prolonged Mediterranean storm system triggered severe flooding, which disrupted communities and transportation networks.

3. Testing Wikimedia API...
   ✓ 2 images found
      File:Collage of 2023 Turkey–Syria earthquake.jpg
      File:2023 Turkey-Syria Earthquakes.svg

✓ All API connections working — ready for Workshop 2.


"Issue 1: Gemini Model Quota Error\nThe original code provided by the instructor used the Gemini 2.0 Flash model. When running the code, the API returned a 429 RESOURCE_EXHAUSTED error because the current API key did not have available quota for the Gemini 2.0 Flash model.\nThe error message showed that the quota limit for the model was 0, which means the request was rejected due to model‑specific quota restrictions rather than a problem with the API key, SDK, or code structure.\nSolution:\nThe model was updated from Gemini 2.0 Flash to Gemini 2.5 Flash . The API key was already verified to have access to Gemini 2.5 models, so switching the model resolved the quota issue while keeping the same Gemini API workflow.\n\nIssue 2: Wikimedia API Endpoint Error\nThe original code used the Wikimedia Commons API endpoint:\nhttps://api.wikimedia.org/core/v1/commons/search/files\nThis endpoint returned errors because the API format and access requirements were different from the current environme

Issue 1: Gemini Model Quota Error
The original code provided by the instructor used the Gemini 2.0 Flash model. When running the code, the API returned a 429 RESOURCE_EXHAUSTED error because the current API key did not have available quota for the Gemini 2.0 Flash model.
The error message showed that the quota limit for the model was 0, which means the request was rejected due to model-specific quota restrictions rather than a problem with the API key, SDK, or code structure.
Solution:
The model was updated from Gemini 2.0 Flash to Gemini 2.5 Flash . The API key was already verified to have access to Gemini 2.5 models, so switching the model resolved the quota issue while keeping the same Gemini API workflow.
Issue 2: Wikimedia API Endpoint Error
The original code used the Wikimedia Commons API endpoint:
https://api.wikimedia.org/core/v1/commons/search/files
This endpoint returned errors because the API format and access requirements were different from the current environment. The issue was related to the API endpoint compatibility rather than the network connection or request logic.
Solution:
The Wikimedia request was changed to use the standard MediaWiki API endpoint:
https://commons.wikimedia.org/w/api.php
The updated request searches the Wikimedia Commons file namespace directly by using the MediaWiki search API. This endpoint works without additional authentication and is more stable for this type of image search task.
After these two adjustments, the Gemini API and Wikimedia API connections were successfully restored, and the environment is ready for Workshop 2.



## Workshop 2 — Intent Clarification Agent
Builds an agent that asks the user targeted questions about a disaster event and returns a structured JSON object with event type, location, time period, and search keywords.

Collect key elements of the target natural disaster through multi-turn dialogue and generate structured queries to provide input for downstream pipelines.

## Step 2 — Define the System Prompt
The system prompt tells the model what its job is and when to stop asking questions.

In [ ]:
SYSTEM_PROMPT = """
You are a disaster event clarification assistant. Your job is to ask the user
targeted questions to identify exactly which natural disaster event they are
referring to.
You need three pieces of information:
1. Event type (e.g. earthquake, flood, wildfire, hurricane)
2. Location (country, region, or city)
3. Time period (year, or approximate date range)
If the user mentions clues about damage, casualties, intensity, or severity,
you should also collect information about the severity of the disaster.
Ask only ONE question per response.
Once you have all the required information, respond ONLY with a JSON object in
this exact format and nothing else:
{
  "event_type": "...",
  "location": "...",
  "time_period": "...",
  "severity": "...",
  "keywords": ["...", "...", "..."]
}
If severity is not mentioned or not relevant, set "severity" to "unknown".
"""
print("✓ System prompt defined.")

✓ System prompt defined.


## Step 3 — Build the Intent Agent
The agent starts a Gemini chat session with the system prompt, loops to read user input and send to Gemini, and stops when Gemini returns a JSON object.

In [ ]:
"""## Step 3 — Build the Intent Agent
The agent starts a Gemini chat session with the system prompt, loops to read user input and send to Gemini.
After model outputs JSON, run Pydantic boundary validation.
If validation fails, stay in conversation loop to collect better user input.
Stop and return validated dict only when JSON passes schema check.
"""
def run_intent_agent():
    # FIX: use standard ASCII hyphens for model name: gemini-2.5-flash
    chat = client.chats.create(
        model="gemini-2.5-flash",
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT
        )
    )
    print("Describe the disaster event you want to know about (type 'quit' to exit):\n")
    while True:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() == "quit":
            print("Exiting.")
            break
        reply = chat.send_message(user_input).text.strip()
        if reply.startswith("```"):
            reply = reply.strip("`").strip()
            if reply.startswith("json"):
                reply = reply[4:].strip()
        if reply.startswith("{"):
            # Agent boundary validation: validate raw json string before returning
            validated_result = validated_intent_agent_output(reply)
            if validated_result is not None:
                print("\n✓ Event identified and passed validation:")
                print(json.dumps(validated_result, indent=2))
                return validated_result
            else:
                # Validation fails: continue loop, keep asking user for better information
                print("\n⚠ Intent output failed validation, continue asking for information.\n")
                continue
        print(f"Agent: {reply}\n")

print("✓ run_intent_agent() defined — run the next cell to start a conversation.")


✓ run_intent_agent() defined — run the next cell to start a conversation.


## Step 4 — Run the Agent
Run the cell below to start a live conversation with your agent.

In [ ]:
event_query = run_intent_agent()

Describe the disaster event you want to know about (type 'quit' to exit):

You:  Tell me about the Wenchuan earthquake in China
Agent: What year did the Wenchuan earthquake occur?

You: 2008

✓ Event identified and passed validation:
{
  "event_type": "Earthquake",
  "location": "Wenchuan, China",
  "time_period": "2008",
  "keywords": [
    "Wenchuan",
    "earthquake",
    "China",
    "2008"
  ]
}




## Workshop 3 — Event Identification and Article Sourcing Agent
Takes the structured event query from Workshop 2 and uses Gemini's built-in Google Search grounding to find real news articles about the event. Returns a structured JSON containing event details and a list of sourced article URLs.

Leveraging Gemini's search grounding capabilities to retrieve information on real-world events through the web, it outputs standardized event details and news sources, pinpointing the specifics of actual events.

## Step 2 — Provide Your Event Query
This is the structured JSON produced by the Intent Clarification Agent in Workshop 2.

In [ ]:
# Replace with your actual output from the Workshop 2 intent agent,
# or use this sample to test the event agent.
# Using event_query from Workshop 2 Intent Clarification Agent
print("✓ Event query ready (from Workshop 2):")
import json
print(json.dumps(event_query, indent=2))

✓ Event query ready (from Workshop 2):
{
  "event_type": "Earthquake",
  "location": "Wenchuan, China",
  "time_period": "2008",
  "keywords": [
    "Wenchuan",
    "earthquake",
    "China",
    "2008"
  ]
}


## Step 3 — Define the System Prompt
The system prompt tells the model what structured JSON it must return and what fields to include.

In [ ]:
SYSTEM_PROMPT = """
You are a disaster event research agent. Given a structured query, use Google
Search to find accurate, current information about the specified natural disaster.
Return ONLY a valid JSON object — no markdown, no extra text — with these fields:
{
  "event_name": "commonly used name for the event",
  "event_type": "type of disaster",
  "location": "specific locations affected",
  "date": "date or date range of the event",
  "casualties": "reported deaths or affected people if available",
  "severity_score": "integer from 1 to 5, 1 = minor disaster with few injuries, 5 = catastrophic disaster with massive deaths and destruction",
  "summary": "2-3 sentence factual overview of the event",
  "articles": [
    {"title": "article headline", "url": "article URL", "snippet": "1 sentence description"},
    ...
  ]
}
Include up to 5 articles. Populate articles from the sources you searched. Calculate severity_score based on reported casualties, damage scale and impact range.
"""
print("✓ System prompt defined.")

✓ System prompt defined.


## Step 4 — Build the Event Identification Agent
Builds a prompt from the event_query fields, calls Gemini with grounding enabled, extracts article URLs from grounding metadata, parses the JSON response, and returns the combined result.

In [ ]:
"""## Step 4 — Build the Event Identification Agent
Builds a prompt from the event_query fields, calls Gemini with grounding enabled, extracts article URLs from grounding metadata.
Add Pydantic agent‑boundary validation before returning event_data.
"""
def run_event_agent(event_query: dict) -> dict:
    # Build prompt
    prompt = f"""
Event search parameters:
event_type: {event_query['event_type']}
location: {event_query['location']}
time_period: {event_query['time_period']}
keywords: {event_query['keywords']}
Use Google Search to get real news data. ONLY output pure JSON, no extra words, no markdown.
"""
    # Call Gemini 2.5 Flash with grounding, FIX: standard ASCII hyphens in model name
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            tools=[types.Tool(google_search=types.GoogleSearch())]
        )
    )
    # Debug: Print raw output text to check if empty
    print("=== Model raw output ===")
    print(repr(response.text))
    print("========================")
    # Multi‑layer empty check to avoid empty string passed to json.loads
    if not response.candidates:
        raise RuntimeError("API returned empty candidates, no response from Gemini")
    raw_text = response.text
    if not raw_text or raw_text.strip() == "":
        raise RuntimeError("Model output is blank, cannot parse JSON")
    # Extract grounding sources
    sources = []
    grounding = response.candidates[0].grounding_metadata
    if grounding and grounding.grounding_chunks:
        for chunk in grounding.grounding_chunks:
            if chunk.web:
                sources.append({"title": chunk.web.title, "url": chunk.web.uri, "snippet": ""})
    # Fixed line: Remove all backticks globally to eliminate markdown code fences
    raw = raw_text.replace("`", "").strip()
    event_data = json.loads(raw)
    # Merge article sources
    if not event_data.get("articles") and sources:
        event_data["articles"] = sources
    elif sources:
        existing_urls = {a["url"] for a in event_data["articles"]}
        for src in sources:
            if src["url"] not in existing_urls:
                event_data["articles"].append(src)

    # -------- Agent boundary validation --------
    # Dump dict to json string and run event schema validation
    validated_event = validated_event_agent_output(json.dumps(event_data))
    return validated_event

print("✓ run_event_agent() defined — run the next cell to call the agent.")


✓ run_event_agent() defined — run the next cell to call the agent.


## Step 5 — Run the Agent
Run the cell below to call your event agent with the query from Step 2.

In [ ]:
event_data = run_event_agent(event_query)
print(json.dumps(event_data, indent=2))

=== Model raw output ===
'{\n  "event_name": "Wenchuan earthquake",\n  "event_type": "Earthquake",\n  "location": "Wenchuan County, Sichuan Province, southwestern China, with the epicenter near Dujiangyan, about 80 km west-northwest of Chengdu. Affected areas included Sichuan, Gansu, and Shaanxi provinces, and tremors were felt across ten provinces.",\n  "date": "May 12, 2008, at 14:28:01 China Standard Time (06:28:01 UTC)",\n  "casualties": "At least 69,185 people killed, 374,171 injured, and 18,467 missing. The final official Chinese government assessment counted almost 90,000 people dead or missing, including over 5,300 children. Over 4.8 million people were left homeless, and more than 45.5 million people were affected across 10 provinces.",\n  "severity_score": 5,\n  "summary": "The Wenchuan earthquake was a catastrophic magnitude 7.9 to 8.0 earthquake that struck Wenchuan County, Sichuan Province, China, on May 12, 2008. The disaster resulted in nearly 90,000 deaths or missing in



## Workshop 4 — Event Image Collection
Takes the event JSON from Workshop 3 and uses the Wikimedia Commons API to find real photographs of the disaster event. The event JSON will contain an "images" list with URLs, thumbnails, metadata, and local downloaded file paths.

Invoke the Wikimedia API to retrieve, filter, and download publicly available disaster-related images, and obtain image metadata, copyright information, and local file paths.

## Step 1 — Install and Import Dependencies
The Wikimedia API only requires the requests library — no API key needed.

In [ ]:
#improve：clean file names
!pip install requests --quiet
import requests
import json
import os

WIKIMEDIA_API_URL = "https://commons.wikimedia.org/w/api.php"

# Wikimedia requires a descriptive User-Agent on every request, or it returns HTTP 403.
# See: https://meta.wikimedia.org/wiki/User-Agent_policy
print("✓ requests, json, os imported. No API key needed for Wikimedia.")
HEADERS = {"User-Agent": "EventInsightBot/1.0 (https://example.com/contact; student-project)"}


def download_image(image_url: str, save_directory: str = "disaster_event_images") -> str:
    """
    Download image from remote Wikimedia URL to local disk.
    Args:
        image_url: remote full image URL from Wikimedia API
        save_directory: folder to store downloaded images
    Returns:
        full local file path string of saved image, empty string if failed
    """
    # Create target directory if it does not exist
    if not os.path.exists(save_directory):
        os.makedirs(save_directory)

    # Extract filename from URL, then strip query parameters (e.g. ?utm_source=...).
    # Key concept: split("?")[0] removes everything after the ? so the saved file has a clean extension.
    file_name = image_url.split("/")[-1]
    file_name = file_name.split("?")[0]
    full_local_path = os.path.join(save_directory, file_name)

    try:
        # stream=True downloads binary content in chunks, efficient for large files.
        download_response = requests.get(image_url, headers=HEADERS, stream=True, timeout=20)
        if download_response.status_code == 200:
            with open(full_local_path, "wb") as output_file:
                output_file.write(download_response.content)
            print(f"Image downloaded successfully to: {full_local_path}")
            return full_local_path
        else:
            print(f"Download failed, HTTP status code: {download_response.status_code}")
            return ""
    except Exception as download_err:
        print(f"Exception occurred during image download: {str(download_err)}")
        return ""


✓ requests, json, os imported. No API key needed for Wikimedia.


## Step 2 — Provide Your Event Data
This is the structured JSON produced by the Event Identification Agent in Workshop 3.

In [ ]:
# This is the structured JSON produced by the Event Identification Agent in Workshop 3.
# Using event_data from Workshop 3 Event Identification Agent
print("✓ Event data ready (from Workshop 3). Will search for images of:")
print(f"  event_name : {event_data['event_name']}")
print(f"  location   : {event_data['location']}")

✓ Event data ready (from Workshop 3). Will search for images of:
  event_name : Wenchuan earthquake
  location   : Wenchuan County, Sichuan Province, southwestern China, with the epicenter near Dujiangyan, about 80 km west-northwest of Chengdu. Affected areas included Sichuan, Gansu, and Shaanxi provinces, and tremors were felt across ten provinces.


## Step 3 — Search for Image Titles (API Call 1)
The Wikimedia Commons API requires two separate calls to go from a search query to image URLs.
This step makes the first call: searching the File namespace (namespace ID = 6) to get a list of relevant file titles.

In [ ]:
#improve:kick tiff files out

import time

MAX_IMAGES = 5

# Extract and clean search terms.
# Key concept: Gemini may return a verbose location sentence — take only the first segment for search.
event_name = event_data["event_name"]
raw_location = event_data["location"]
location = raw_location.split(",")[0].strip()  # e.g. "Noto Peninsula, Ishikawa..." → "Noto Peninsula"
event_type = event_data.get("event_type", "disaster")

# Wikimedia ANDs every term — fewer words = broader results.
# Start specific, fall back to broader queries if nothing is found.
search_queries = [
    f"{event_name} {location}",
    f"{event_name}",
    f"{location} {event_type}",
]

file_titles = []
for i, query in enumerate(search_queries):
    search_params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "srnamespace": 6,
        # Request extra results because we filter out .svg AND .tiff (satellite images, hundreds of MB each).
        "srlimit": MAX_IMAGES * 3,
        "format": "json"
    }

    search_resp = requests.get(url=WIKIMEDIA_API_URL, params=search_params, headers=HEADERS)
    print(f"Query {i+1}: '{query}' → HTTP {search_resp.status_code}")

    if search_resp.status_code == 429:
        # Rate limited — wait 2 seconds and retry once.
        print("  Rate limited, waiting 2s before retry...")
        time.sleep(2)
        search_resp = requests.get(url=WIKIMEDIA_API_URL, params=search_params, headers=HEADERS)
        print(f"  Retry → HTTP {search_resp.status_code}")

    if search_resp.status_code != 200:
        print(f"  Skipping: {search_resp.text[:150]}")
        time.sleep(1)
        continue

    search_data = search_resp.json()
    total_hits = search_data.get("query", {}).get("searchinfo", {}).get("totalhits", 0)
    print(f"  Total hits: {total_hits}")

    if total_hits > 0:
        # Filter out vector graphics (.svg) and large satellite raster files (.tiff/.tif).
        # Key concept: tuple in endswith() checks multiple extensions in one pass.
        file_titles = [
            hit["title"]
            for hit in search_data.get("query", {}).get("search", [])
            if not hit["title"].lower().endswith((".svg", ".tiff", ".tif"))
        ][:MAX_IMAGES]
        if file_titles:
            break

    time.sleep(1)  # polite delay between queries to avoid 429

print(f"\n✓ Found {len(file_titles)} file titles:")
for t in file_titles:
    print(f"  {t}")


Query 1: 'Wenchuan earthquake Wenchuan County' → HTTP 200
  Total hits: 16

✓ Found 5 file titles:
  File:1 wenchuan county sichuan panorama 2013.jpg
  File:Cemetery of Victims of Wenchuan Earthquake in Beichuan County.jpg
  File:Wenchuan.Qiang.diaolou.jpg
  File:Wenchuan.pont.detruit.jpg
  File:Baihua Bridge.jpg


## Step 4 — Fetch Image URLs and Metadata (API Call 2)
Now that you have a list of file titles, make the second API call to retrieve the full image URLs and metadata (description, license, author) for each file.

In [ ]:
info_params = {
    "action": "query",
    "titles": "|".join(file_titles),
    "prop": "imageinfo",
    "iiprop": "url|extmetadata",
    "iiurlwidth": 800,
    "format": "json"
}
info_resp = requests.get(url=WIKIMEDIA_API_URL, params=info_params, headers=HEADERS)
info_data = info_resp.json()
# Initialize empty list to store structured image metadata dicts
images = []
# Iterate all returned page objects from API response
for page in info_data.get("query", {}).get("pages", {}).values():
    # Skip page entries without valid image data
    if "imageinfo" not in page:
        continue
    # Extract primary image info block
    info = page["imageinfo"][0]
    # Extract extended license/author/description metadata
    ext = info.get("extmetadata", {})
    # Get remote image url for download
    remote_img_url = info.get("url", "")
    # Download image and get local save path
    local_file_path = download_image(remote_img_url)
    # Append standardized image object to list with local file path
    images.append({
        "title": page["title"],
        "url": remote_img_url,
        "thumbnail_url": info.get("thumburl", remote_img_url),
        "description": ext.get("ImageDescription", {}).get("value", ""),
        "license": ext.get("LicenseShortName", {}).get("value", ""),
        "author": ext.get("Artist", {}).get("value", ""),
        "local_path": local_file_path
    })
print(f"✓ Retrieved metadata for {len(images)} image(s).")
for img in images:
    print(f"  {img['title']}  ({img['license']}) | Local path: {img['local_path']}")

Image downloaded successfully to: disaster_event_images/1_wenchuan_county_sichuan_panorama_2013.jpg
Image downloaded successfully to: disaster_event_images/Baihua_Bridge.jpg
Image downloaded successfully to: disaster_event_images/Cemetery_of_Victims_of_Wenchuan_Earthquake_in_Beichuan_County.jpg
Image downloaded successfully to: disaster_event_images/Wenchuan.Qiang.diaolou.jpg
Image downloaded successfully to: disaster_event_images/Wenchuan.pont.detruit.jpg
✓ Retrieved metadata for 5 image(s).
  File:1 wenchuan county sichuan panorama 2013.jpg  (CC BY-SA 3.0) | Local path: disaster_event_images/1_wenchuan_county_sichuan_panorama_2013.jpg
  File:Baihua Bridge.jpg  (CC BY-SA 3.0) | Local path: disaster_event_images/Baihua_Bridge.jpg
  File:Cemetery of Victims of Wenchuan Earthquake in Beichuan County.jpg  (CC BY-SA 4.0) | Local path: disaster_event_images/Cemetery_of_Victims_of_Wenchuan_Earthquake_in_Beichuan_County.jpg
  File:Wenchuan.Qiang.diaolou.jpg  (CC BY-SA 3.0) | Local path: disas

## Step 5 — Wrap Into Reusable Functions
Combines the logic from Steps 3 and 4 into two clean, reusable functions.

In [ ]:
"""## Step 5 — Wrap Into Reusable Functions
Combines the logic from Steps 3 and 4 into two clean, reusable functions.
Add image‑list boundary validation using validate_images_list.
FIX: pass event_type as function parameter instead of reading global variable
"""
import time
def search_event_images(event_name: str, location: str, event_type: str, max_images: int = 5) -> list:
    """
    Complete Wikimedia search pipeline: search titles → fetch metadata → download images
    Args:
        event_name: name of disaster event
        location: geographic location of disaster
        event_type: type of disaster (e.g. earthquake, flood)
        max_images: maximum valid images to return
    Returns:
        List of raw image dicts containing metadata and local file path
    """
    # Multiple fallback search queries (same logic as Step3)
    search_queries = [
        f"{event_name} {location}",
        f"{event_name}",
        f"{location} {event_type}"
    ]
    file_titles = []
    for query in search_queries:
        search_params = {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srnamespace": 6,
            "srlimit": max_images * 3,
            "format": "json"
        }
        resp = requests.get(WIKIMEDIA_API_URL, params=search_params, headers=HEADERS)
        # Handle rate‑limit 429
        if resp.status_code == 429:
            time.sleep(2)
            resp = requests.get(WIKIMEDIA_API_URL, params=search_params, headers=HEADERS)
        if resp.status_code != 200:
            time.sleep(1)
            continue
        data = resp.json()
        total_hits = data.get("query", {}).get("searchinfo", {}).get("totalhits", 0)
        if total_hits > 0:
            # Filter out svg, tiff, tif files
            file_titles = [
                hit["title"]
                for hit in data.get("query", {}).get("search", [])
                if not hit["title"].lower().endswith((".svg", ".tiff", ".tif"))
            ][:max_images]
            if file_titles:
                break
        time.sleep(1)
    if not file_titles:
        return []
    # Second API call: get full image info and urls
    info_params = {
        "action": "query",
        "titles": "|".join(file_titles),
        "prop": "imageinfo",
        "iiprop": "url|extmetadata",
        "iiurlwidth": 800,
        "format": "json"
    }
    info_resp = requests.get(WIKIMEDIA_API_URL, params=info_params, headers=HEADERS)
    if info_resp.status_code != 200:
        print(f"Wikimedia imageinfo failed: HTTP {info_resp.status_code}")
        print(f"Response body: {info_resp.text[:500]}")
        return []
    info_data = info_resp.json()
    output_images = []
    for page in info_data.get("query", {}).get("pages", {}).values():
        if "imageinfo" not in page:
            continue
        info = page["imageinfo"][0]
        ext_meta = info.get("extmetadata", {})
        img_url = info.get("url", "")
        local_path = download_image(img_url)
        output_images.append({
            "title": page["title"],
            "url": img_url,
            "thumbnail_url": info.get("thumburl", img_url),
            "description": ext_meta.get("ImageDescription", {}).get("value", ""),
            "license": ext_meta.get("LicenseShortName", {}).get("value", ""),
            "author": ext_meta.get("Artist", {}).get("value", ""),
            "local_path": local_path
        })
    return output_images

def collect_event_images(event_data: dict, max_images: int = 5) -> dict:
    """
    Pipeline‑facing wrapper required by boilerplate.
    Adds "images" key to event_data dict.
    Run image‑list validation: discard malformed image entries.
    Args:
        event_data: event dict from Workshop3
        max_images: maximum valid photos
    Returns:
        Enriched event dict with validated images list attached
    """
    event_name = event_data["event_name"]
    raw_location = event_data["location"]
    location = raw_location.split(",")[0].strip()
    event_type = event_data["event_type"]
    # Pass event_type as parameter, no global variable dependency
    image_results = search_event_images(event_name, location, event_type, max_images)
    # Agent boundary validation for image list
    validated_image_list = validate_images_list(image_results)
    event_data["images"] = validated_image_list
    return event_data

print("✓ search_event_images() and collect_event_images() defined — run the next cell to start image collection.")


✓ search_event_images() and collect_event_images() defined — run the next cell to start image collection.


## Step 6 — Run End-to-End
Run the cell below to call collect_event_images() and print the enriched event JSON.

In [ ]:
# Workshop4 Step6: use required collect_event_images wrapper function
enriched_data = collect_event_images(event_data)
print(json.dumps(enriched_data, indent=2))

Image downloaded successfully to: disaster_event_images/1_wenchuan_county_sichuan_panorama_2013.jpg
Image downloaded successfully to: disaster_event_images/Baihua_Bridge.jpg
Image downloaded successfully to: disaster_event_images/Cemetery_of_Victims_of_Wenchuan_Earthquake_in_Beichuan_County.jpg
Image downloaded successfully to: disaster_event_images/Wenchuan.Qiang.diaolou.jpg
Image downloaded successfully to: disaster_event_images/Wenchuan.pont.detruit.jpg
{
  "event_name": "Wenchuan earthquake",
  "event_type": "Earthquake",
  "location": "Wenchuan County, Sichuan Province, southwestern China, with the epicenter near Dujiangyan, about 80 km west-northwest of Chengdu. Affected areas included Sichuan, Gansu, and Shaanxi provinces, and tremors were felt across ten provinces.",
  "date": "May 12, 2008, at 14:28:01 China Standard Time (06:28:01 UTC)",
  "casualties": "At least 69,185 people killed, 374,171 injured, and 18,467 missing. The final official Chinese government assessment counte

## Workshop 5 - Summarization Agent

## Workshop 5 — Summarization Agent
Reads the article snippets from the event JSON produced in Workshop 3 and uses Gemini to generate a structured "situation_brief" — the human-readable core of the final deliverable.

Utilizes **generative summarization** to integrate event metadata and news source material, producing structured disaster situation briefings—distinct from extractive summarization, which merely copies sentences from the original text.

## Step 2 — Provide Your Event Data
This is the event JSON produced after Workshop 3 (event details + articles) and Workshop 4 (images).

In [ ]:
# Using event_data from Workshop 4 (contains event details + articles + images)
print(f"✓ Loaded event data: {event_data['event_name']}")
print(f"  Article count: {len(event_data['articles'])} | Image count: {len(event_data['images'])}")

✓ Loaded event data: Wenchuan earthquake
  Article count: 5 | Image count: 5


## Step 3 — Define the System Prompt
The system prompt defines the exact JSON schema the model must return for the situation brief.

In [ ]:
SYSTEM_PROMPT = """
You are a disaster summarization agent. Synthesize article snippets into structured JSON brief.
Return ONLY valid JSON without markdown fences or extra text, follow schema strictly:
{
  "headline": "Single-sentence headline, max 20 words",
  "overview": "2-3 factual sentences describing event time/location/cause",
  "severity": "2-3 sentences for casualties, damage magnitude",
  "affected_population": "1-2 sentences on impacted residents",
  "response": "1-2 sentences for government/international relief actions",
  "key_facts": ["3 to 5 short standalone factual statements about the disaster"],
  "sources_used": ["article_url_1", "article_url_2"]
}
Fill missing minor details with verified public knowledge of this disaster.
List all input article URLs in sources_used.
Generate exactly 3-5 entries inside key_facts array.
"""
print("✓ System prompt loaded, fixed JSON schema defined")

✓ System prompt loaded, fixed JSON schema defined


## Step 4 — Build the Article Context Helper
Before calling Gemini, format the articles list into a readable text block.

In [ ]:
def build_article_context(articles: list) -> str:
    if not articles:
        return "No article snippets available."
    output_lines = []
    for idx, article in enumerate(articles, start=1):
        output_lines.append(f"Article {idx}: {article.get('title', 'Untitled')}")
        output_lines.append(f"URL: {article.get('url', '')}")
        output_lines.append(f"Snippet: {article.get('snippet', '')}")
        output_lines.append("")
    return "\n".join(output_lines)

# Test helper function
print("\n=== Formatted Article Context Preview ===")
print(build_article_context(event_data["articles"]))
print("=========================================\n")


=== Formatted Article Context Preview ===
Article 1: Sichuan earthquake of 2008 | Overview, Damage, & Facts - Britannica
URL: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH37BxzSRjGUAaAqps01UVrZL9s1f5iubiQ4Nhho8b5qs-am4mFPxkhgcCTJrl8ApxK5JkPOn7YOZJQy5hX1U2MKlJbvzGo6k-9tvMl0IIGvdhzCZ_lyPkLO_jHQtuwJNgVUvmyVWFRYKjsCT7H2knXCrskUz0=
Snippet: The Sichuan earthquake of 2008, also known as the Wenchuan earthquake, was a massive and enormously devastating earthquake that occurred on May 12, 2008, causing widespread destruction and nearly 90,000 deaths or missing persons.

Article 2: 2008 Sichuan earthquake - Wikipedia
URL: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFRh-QsiOWREgov7levenhBmGEBJUfF1X29i29Fk5aCFXoxN1_7Lpcyx1Gv4tyEeVN0zqcqbHRD3a9EL0UonADlz1tcOJdZMIHxVCBsKWbFEBPU1ZF0Fp9qMf5VwGaFwdRhQSsbJa3xZu6yb_JbHSo=
Snippet: The 2008 Sichuan earthquake, with a magnitude of 8.0 Ms, occurred on May 12, 2008, in Wenchuan County, resulting in over 69,

## Step 5 — Build the Summarization Agent
Calls build_article_context() to format the articles, builds a prompt with event metadata and article context, calls Gemini with system_instruction (no grounding needed), strips markdown code fences, parses the JSON response, and adds situation_brief to the event_data dict.

In [ ]:
"""## Step 5 — Build the Summarization Agent
Calls build_article_context() to format the articles, builds a prompt with event metadata and article context, calls Gemini with system_instruction (no grounding needed).
Run Pydantic validation for situation brief as agent boundary check.
"""
def run_summarization_agent(event_input: dict) -> dict:
    # Format article text context
    article_text = build_article_context(event_input["articles"])
    # Build full user prompt with core event metadata + article references
    user_prompt = f"""
Disaster Event Metadata:
Event Name: {event_input['event_name']}
Location: {event_input['location']}
Date: {event_input['date']}
Casualty Summary: {event_input['casualties']}
Reference Article Snippets:
{article_text}
Generate structured situation brief following the system prompt JSON schema.
"""
    # FIX: standard ASCII hyphens for model name gemini-2.5-flash
    llm_response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=user_prompt,
        config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT)
    )
    # Clean markdown code block markers before JSON parse
    raw_output = llm_response.text.strip()
    if raw_output.startswith("```"):
        raw_output = raw_output.strip("`").strip()
        if raw_output.startswith("json"):
            raw_output = raw_output[4:].strip()

    # Agent boundary validation for summarization output
    validated_brief = validated_situation_brief(raw_output)
    event_input["situation_brief"] = validated_brief
    return event_input

print("✓ run_summarization_agent() defined — run the next cell to generate the brief.")


✓ run_summarization_agent() defined — run the next cell to generate the brief.


## Step 6 — Run the Agent and Validate the Output
Run the cell below to generate the situation brief.

In [ ]:
final_event = run_summarization_agent(event_data)
# Print complete enriched JSON
print("=== Full Enriched Event JSON (with situation_brief) ===")
print(json.dumps(final_event, indent=2))
# Validate all required brief fields exist (added key_facts)
required_brief_fields = ["headline", "overview", "severity", "affected_population", "response", "key_facts", "sources_used"]
brief_data = final_event.get("situation_brief", {})
missing_fields = [field for field in required_brief_fields if field not in brief_data]
if missing_fields:
    print(f"\n⚠ Missing required brief fields: {missing_fields}")
else:
    print("\n✓ All mandatory situation brief fields generated successfully")

=== Full Enriched Event JSON (with situation_brief) ===
{
  "event_name": "Wenchuan earthquake",
  "event_type": "Earthquake",
  "location": "Wenchuan County, Sichuan Province, southwestern China, with the epicenter near Dujiangyan, about 80 km west-northwest of Chengdu. Affected areas included Sichuan, Gansu, and Shaanxi provinces, and tremors were felt across ten provinces.",
  "date": "May 12, 2008, at 14:28:01 China Standard Time (06:28:01 UTC)",
  "casualties": "At least 69,185 people killed, 374,171 injured, and 18,467 missing. The final official Chinese government assessment counted almost 90,000 people dead or missing, including over 5,300 children. Over 4.8 million people were left homeless, and more than 45.5 million people were affected across 10 provinces.",
  "summary": "The Wenchuan earthquake was a catastrophic magnitude 7.9 to 8.0 earthquake that struck Wenchuan County, Sichuan Province, China, on May 12, 2008. The disaster resulted in nearly 90,000 deaths or missing in

In [ ]:
# final output
# Final formatted output: aggregates all pipeline results into one readable report.
# Key concept: dict.get() with defaults prevents KeyError if a pipeline step returned fewer fields than expected.

def print_final_report(event: dict) -> None:
    """Print a structured situation brief combining all agent outputs."""

    # --- Header: core event metadata from Workshop 3 ---
    print("=" * 70)
    print(f"  EVENT INSIGHT — FINAL SITUATION REPORT")
    print("=" * 70)
    print(f"  Event    : {event.get('event_name', 'N/A')}")
    print(f"  Type     : {event.get('event_type', 'N/A')}")
    print(f"  Location : {event.get('location', 'N/A')}")
    print(f"  Date     : {event.get('date', 'N/A')}")
    print(f"  Severity : {event.get('casualties', 'N/A')}  (score: {event.get('severity_score', 'N/A')}/5)")
    print()

    # --- Overview: short factual summary from Workshop 3 ---
    print("-" * 70)
    print("  OVERVIEW")
    print("-" * 70)
    print(f"  {event.get('summary', 'N/A')}")
    print()

    # --- Situation Brief: generated by Workshop 5 summarization agent ---
    brief = event.get("situation_brief", {})
    if brief:
        print("-" * 70)
        print("  SITUATION BRIEF")
        print("-" * 70)
        print(f"  Headline : {brief.get('headline', 'N/A')}")
        print(f"  Overview : {brief.get('overview', 'N/A')}")
        print(f"  Severity : {brief.get('severity', 'N/A')}")
        print(f"  Affected : {brief.get('affected_population', 'N/A')}")
        print(f"  Response : {brief.get('response', 'N/A')}")
        # Key facts: list comprehension iterates the array and numbers each entry.
        facts = brief.get("key_facts", [])
        if facts:
            print("  Key Facts:")
            for i, fact in enumerate(facts, 1):
                print(f"    {i}. {fact}")
        print()

    # --- Articles: sourced by Workshop 3 event identification agent ---
    articles = event.get("articles", [])
    if articles:
        print("-" * 70)
        print(f"  SOURCED ARTICLES  ({len(articles)} found)")
        print("-" * 70)
        for idx, art in enumerate(articles, 1):
            print(f"  [{idx}] {art.get('title', 'Untitled')}")
            print(f"      URL    : {art.get('url', 'N/A')}")
            print(f"      Snippet: {art.get('snippet', 'N/A')}")
            print()

    # --- Images: collected by Workshop 4 image collector ---
    images = event.get("images", [])
    if images:
        print("-" * 70)
        print(f"  COLLECTED IMAGES  ({len(images)} found)")
        print("-" * 70)
        for idx, img in enumerate(images, 1):
            print(f"  [{idx}] {img.get('title', 'Untitled')}")
            print(f"      URL       : {img.get('url', 'N/A')}")
            print(f"      Thumbnail : {img.get('thumbnail_url', 'N/A')}")
            print(f"      License   : {img.get('license', 'N/A')}")
            print(f"      Author    : {img.get('author', 'N/A')}")
            print(f"      Local file: {img.get('local_path', 'N/A')}")
            print()

    print("=" * 70)
    print("  END OF REPORT")
    print("=" * 70)


# Run the final report using the enriched event data from the pipeline.
print_final_report(final_event)


  EVENT INSIGHT — FINAL SITUATION REPORT
  Event    : Wenchuan earthquake
  Type     : Earthquake
  Location : Wenchuan County, Sichuan Province, southwestern China, with the epicenter near Dujiangyan, about 80 km west-northwest of Chengdu. Affected areas included Sichuan, Gansu, and Shaanxi provinces, and tremors were felt across ten provinces.
  Date     : May 12, 2008, at 14:28:01 China Standard Time (06:28:01 UTC)
  Severity : At least 69,185 people killed, 374,171 injured, and 18,467 missing. The final official Chinese government assessment counted almost 90,000 people dead or missing, including over 5,300 children. Over 4.8 million people were left homeless, and more than 45.5 million people were affected across 10 provinces.  (score: 5/5)

----------------------------------------------------------------------
  OVERVIEW
----------------------------------------------------------------------
  The Wenchuan earthquake was a catastrophic magnitude 7.9 to 8.0 earthquake that struck W

If cells are not run sequentially from top‑to‑bottom, re‑running isolated cells can trigger undefined‑variable or stale‑data issues. This is a known Colab notebook limitation, not a code bug. The full end‑to‑end pipeline works correctly when executing all cells top‑down.

## Workshop 6 - Agent Validation with Pydantic

The former verification component has now been fully integrated into the preceding steps, enabling verification to take place during execution.

Defines Pydantic schemas for each agent's output and adds model_validate() calls so that structural errors are caught immediately at the point of production.

Perform model validation on the output of each agent in the pipeline—checking fields, data types, and custom constraints—to intercept invalid data and ensure that the data passed to the next stage is valid and compliant.

## Step 1 — Install Pydantic [  full imports]
Pydantic is the validation library we will use throughout this workshop.
No API key is needed for anything in this notebook.

In [ ]:
!pip install pydantic --quiet
import pydantic
from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import List
import typing_extensions as te
print(f"✓ pydantic {pydantic.__version__} ready.")

✓ pydantic 2.13.4 ready.


## Step 2 — Define Pydantic Schemas for Every Agent
Each agent in the pipeline has an expected output shape. We define that shape as a Pydantic BaseModel.

In [ ]:
class IntentOutput(BaseModel):
    """Schema for Intent Clarification Agent output."""
    event_type: str
    location: str
    time_period: str
    # min_length=1 enforces non‑empty list
    keywords: List[str] = Field(min_length=1)

class ArticleItem(BaseModel):
    """Nested schema: single news article inside EventOutput."""
    title: str
    url: str
    snippet: str

class EventOutput(BaseModel):
    """Schema for Event Identification Agent output."""
    event_name: str
    event_type: str
    location: str
    date: str
    casualties: str
    summary: str
    severity_score: int  # ← ：W3  1-5 severity_score
    # Nested list: Pydantic recursively validates every ArticleItem
    articles: List[ArticleItem]

class ImageItem(BaseModel):
    """Schema for one collected image record."""
    title: str
    url: str
    thumbnail_url: str
    description: str
    license: str
    author: str
    local_path: str  # ← ：W4 picture local_path

class SituationBrief(BaseModel):
    """Schema for Summarization Agent output."""
    headline: str
    overview: str
    severity: str
    affected_population: str
    response: str
    key_facts: List[str]  # ← ：W5  3-5 key_facts
    # Require at least one source entry
    sources_used: List[str] = Field(min_length=1)

    @field_validator("headline")
    @classmethod
    def headline_max_20_words(cls, v: str) -> str:
        """Custom validator: reject headline longer than 20 words."""
        word_count = len(v.split())
        if word_count > 20:
            raise ValueError(f"Headline exceeds max 20 words, got {word_count}")
        return v

print("✓ Schemas defined.")


✓ Schemas defined.


## Step 3 — Verify Schemas with model_validate()
Call model_validate() on each schema using the sample dicts below.

In [ ]:
SAMPLE_INTENT = event_query
SAMPLE_EVENT = event_data
SAMPLE_IMAGE = event_data["images"][0] if event_data.get("images") else {}
SAMPLE_BRIEF = final_event["situation_brief"]

def validate_all_schemas():
    """Test each schema against sample dict, print pass/fail status."""
    pairs = [
        (IntentOutput,    SAMPLE_INTENT),
        (EventOutput,     SAMPLE_EVENT),
        (ImageItem,       SAMPLE_IMAGE),
        (SituationBrief,  SAMPLE_BRIEF),
    ]
    for schema_class, sample in pairs:
        try:
            schema_class.model_validate(sample)
            print(f"✓ {schema_class.__name__} valid")
        except ValidationError as e:
            print(f"✗ {schema_class.__name__} failed: {e}")

validate_all_schemas()

✓ IntentOutput valid
✓ EventOutput valid
✓ ImageItem valid
✓ SituationBrief valid


## Step 4 — Deliberately Trigger a ValidationError
Understanding what a ValidationError looks like helps you debug failures when they happen inside your agents.

In [ ]:
# Test 1 — missing required field
bad_brief_missing_field = {
    "headline": "Earthquake hits Turkey",
    "overview": "A 7.8 earthquake struck southeastern Turkey.",
    "severity": "59,000 deaths reported.",
    "affected_population": "13.5 million people affected.",
    "response": "International aid deployed.",
    # sources_used intentionally omitted
}
print("\n--- Test1: bad brief missing sources_used ---")
try:
    SituationBrief.model_validate(bad_brief_missing_field)
except ValidationError as e:
    print(e.errors())

# Test 2 — headline exceeds 20 words
bad_brief_long_headline = {**SAMPLE_BRIEF,
    "headline": "This is an extremely long headline that has way more than twenty words in it and should definitely fail the word count validator we wrote"
}
print("\n--- Test2: bad brief headline >20 words ---")
try:
    SituationBrief.model_validate(bad_brief_long_headline)
except ValidationError as e:
    print(e.errors())


--- Test1: bad brief missing sources_used ---
[{'type': 'missing', 'loc': ('key_facts',), 'msg': 'Field required', 'input': {'headline': 'Earthquake hits Turkey', 'overview': 'A 7.8 earthquake struck southeastern Turkey.', 'severity': '59,000 deaths reported.', 'affected_population': '13.5 million people affected.', 'response': 'International aid deployed.'}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}, {'type': 'missing', 'loc': ('sources_used',), 'msg': 'Field required', 'input': {'headline': 'Earthquake hits Turkey', 'overview': 'A 7.8 earthquake struck southeastern Turkey.', 'severity': '59,000 deaths reported.', 'affected_population': '13.5 million people affected.', 'response': 'International aid deployed.'}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]

--- Test2: bad brief headline >20 words ---
[{'type': 'value_error', 'loc': ('headline',), 'msg': 'Value error, Headline exceeds max 20 words, got 25', 'input': 'This is an extremely long headline that has way 

## Step 5 — Add Validation to the Intent and Event Agents

In [ ]:
def validated_intent_agent_output(raw_json_str: str):
    """Parse and validate the intent agent's JSON output.
    Returns a dict on success, None if validation fails (caller should re-prompt).
    """
    try:
        data = json.loads(raw_json_str)
        validated = IntentOutput.model_validate(data)
        return validated.model_dump()
    except ValidationError as e:
        print(f"Intent validation error: {e}")
        return None

def validated_event_agent_output(raw_json_str: str) -> dict:
    """Parse and validate the event agent's JSON output.
    Raises ValueError if validation fails — a bad event output is unrecoverable.
    """
    raw = raw_json_str.strip()
    # Remove ```json / ``` markdown code block markers
    if raw.startswith("```"):
        raw = raw.removeprefix("```").removesuffix("```").strip()
        if raw.startswith("json"):
            raw = raw.removeprefix("json").strip()
    try:
        data = json.loads(raw)
        validated = EventOutput.model_validate(data)
        return validated.model_dump()
    except ValidationError as e:
        raise ValueError(f"Event agent validation failed: {e}")

# --- Quick smoke test ---
good_intent  = json.dumps(SAMPLE_INTENT)
bad_intent   = json.dumps({"event_type": "earthquake"})  # missing fields
print("Good intent:", validated_intent_agent_output(good_intent))
print("Bad intent: ", validated_intent_agent_output(bad_intent))
print("\nGood event:", "PASS" if validated_event_agent_output(json.dumps(SAMPLE_EVENT)) else "FAIL")
try:
    validated_event_agent_output(json.dumps({"event_name": "test"}))
except ValueError as e:
    print("Bad event (expected error):", str(e)[:80])

Good intent: {'event_type': 'Earthquake', 'location': 'Wenchuan, China', 'time_period': '2008', 'keywords': ['Wenchuan', 'earthquake', 'China', '2008']}
Intent validation error: 3 validation errors for IntentOutput
location
  Field required [type=missing, input_value={'event_type': 'earthquake'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
time_period
  Field required [type=missing, input_value={'event_type': 'earthquake'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
keywords
  Field required [type=missing, input_value={'event_type': 'earthquake'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
Bad intent:  None

Good event: PASS
Bad event (expected error): Event agent validation failed: 7 validation errors for EventOutput
event_type
  


## Step 6 — Add Validation to the Image Collector

In [ ]:
def validated_image_item(raw_image_dict: dict):
    """Validate a single image dict against the ImageItem schema.
    Returns a validated dict on success, None on failure.
    """
    try:
        validated = ImageItem.model_validate(raw_image_dict)
        return validated.model_dump()
    except ValidationError as e:
        print(f"Warning: skip invalid image, error: {e}")
        return None

def validate_images_list(images: list) -> list:
    """Validate a list of raw image dicts, skipping any that fail."""
    output = []
    for img in images:
        res = validated_image_item(img)
        if res is not None:
            output.append(res)
    return output

# --- Quick smoke test ---
good_images = [SAMPLE_IMAGE]
bad_image   = {"title": "File:Missing.jpg"}  # missing required fields
result = validate_images_list([SAMPLE_IMAGE, bad_image])
print(f"Validated {len(result)} of 2 images (1 bad image should be skipped).")

url
  Field required [type=missing, input_value={'title': 'File:Missing.jpg'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
thumbnail_url
  Field required [type=missing, input_value={'title': 'File:Missing.jpg'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
description
  Field required [type=missing, input_value={'title': 'File:Missing.jpg'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
license
  Field required [type=missing, input_value={'title': 'File:Missing.jpg'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
author
  Field required [type=missing, input_value={'title': 'File:Missing.jpg'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
local_path
  Field required [type=missing, input_value={'title': 'File:Missing.jpg'}, input_type=dict]
    For

## Step 7 — Add Validation to the Summarization Agent

In [ ]:
def validated_situation_brief(raw_json_str: str) -> dict:
    """Parse and validate the summarization agent's JSON output.
    Raises ValueError if validation fails.
    """
    raw = raw_json_str.strip()
    if raw.startswith("```"):
        raw = raw.removeprefix("```").removesuffix("```").strip()
        if raw.startswith("json"):
            raw = raw.removeprefix("json").strip()
    try:
        data = json.loads(raw)
        validated = SituationBrief.model_validate(data)
        return validated.model_dump()
    except ValidationError as e:
        raise ValueError(f"SituationBrief validation failed: {e}")

# --- Bonus: response_schema config for Gemini (no live call needed) ---
class SituationBriefSchema(te.TypedDict):
    headline: str
    overview: str
    severity: str
    affected_population: str
    response: str
    sources_used: list[str]

# Example Gemini config snippet (no live api call)
# from google.genai import types
# config = types.GenerateContentConfig(
#     response_mime_type="application/json",
#     response_schema=SituationBriefSchema)

# --- Quick smoke test ---
good_brief_json = json.dumps(SAMPLE_BRIEF)
print("Good brief:", "PASS" if validated_situation_brief(good_brief_json) else "FAIL")
try:
    validated_situation_brief(json.dumps({"headline": "Test"}))
except ValueError as e:
    print("Bad brief (expected error):", str(e)[:80])

"""
---
## References
### Google Gemini
| Resource | Link |
|---|---|
| Gemini API — Text generation | https://ai.google.dev/gemini-api/docs/text-generation |
| Gemini API — Structured output | https://ai.google.dev/gemini-api/docs/structured-output |
| Gemini API — System instructions | https://ai.google.dev/gemini-api/docs/system-instructions |
| google-genai Python SDK | https://pypi.org/project/google-genai/ |
### Wikimedia
| Resource | Link |
|---|---|
| Wikimedia API — Getting started | https://api.wikimedia.org/wiki/Getting_started_with_Wikimedia_APIs |
| MediaWiki API — query action | https://www.mediawiki.org/wiki/API:Query |
| MediaWiki API — imageinfo property | https://www.mediawiki.org/wiki/API:Imageinfo |
### Pydantic
| Resource | Link |
|---|---|
| Pydantic documentation | https://docs.pydantic.dev |
| Pydantic — BaseModel | https://docs.pydantic.dev/latest/concepts/models/ |
| Pydantic — Field constraints | https://docs.pydantic.dev/latest/concepts/fields/ |
| Pydantic — field_validator | https://docs.pydantic.dev/latest/concepts/validators/ |
| typing_extensions — TypedDict | https://typing-extensions.readthedocs.io/ |
"""

Good brief: PASS
Bad brief (expected error): SituationBrief validation failed: 6 validation errors for SituationBrief
overvie


'\n---\n## References\n### Google Gemini\n| Resource | Link |\n|---|---|\n| Gemini API — Text generation | https://ai.google.dev/gemini-api/docs/text-generation |\n| Gemini API — Structured output | https://ai.google.dev/gemini-api/docs/structured-output |\n| Gemini API — System instructions | https://ai.google.dev/gemini-api/docs/system-instructions |\n| google-genai Python SDK | https://pypi.org/project/google-genai/ |\n### Wikimedia\n| Resource | Link |\n|---|---|\n| Wikimedia API — Getting started | https://api.wikimedia.org/wiki/Getting_started_with_Wikimedia_APIs |\n| MediaWiki API — query action | https://www.mediawiki.org/wiki/API:Query |\n| MediaWiki API — imageinfo property | https://www.mediawiki.org/wiki/API:Imageinfo |\n### Pydantic\n| Resource | Link |\n|---|---|\n| Pydantic documentation | https://docs.pydantic.dev |\n| Pydantic — BaseModel | https://docs.pydantic.dev/latest/concepts/models/ |\n| Pydantic — Field constraints | https://docs.pydantic.dev/latest/concepts/fi